In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from sklearnex import patch_sklearn
patch_sklearn()
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from networks import SupervisedManifoldVAE, GatedSpectralAttention, EnhancedSpatialAttention
from sklearn.preprocessing import StandardScaler
from reader import prepare_qsm_dataset
from util import seed_everything, get_m, RandomMasking
from util import mask_crop as mask_crop_fn
from train import get_mixed_data, calibrate_balanced, get_anatomy_only
import copy

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

EXP_NAME = "msw_to_chh_aligned_6d"
N_PCA_COMPONENTS = 64
AUG_FACTOR = 5 
NEG_OVERSAMPLE_FACTOR = 12 
N_FOLDS = 5 
seed_everything(0)

UNLABELED_CHH_PATH = "/media/mts_dbs/chh/nii/q" 

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.1))], p=0.5),
    RandomMasking(p=0.4)
])

# ============================================================
# DATA LOADING & ALIGNED PRE-PROCESSING
# ============================================================

dataset_msw = prepare_qsm_dataset(
    'MSW', 
    '/media/mts_dbs/dbs/all/nii/qsm_115/im', 
    '/media/mts_dbs/dbs/all/nii/seg_ps/',
    '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', 
    'msw_cache_6d_cv.pt', 
    load_cache=True, 
    mask_crop_fn=mask_crop_fn,
    cv_pad=False,
    debug=False
)

dataset_chh = prepare_qsm_dataset(
    'CHH', 
    '/media/mts_dbs/chh/nii/qsm/',
    '/media/mts_dbs/chh/nii/roi/',
    '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv',
    'chh_cache_6d_cv.pt', 
    load_cache=True, 
    mask_crop_fn=mask_crop_fn, 
    unlabeled_nii_path=UNLABELED_CHH_PATH,
    cv_pad=False,
    debug=False
)

# 1. Get RAW (unaugmented) data for PCA fitting and unlabeled sets
X_msw_raw_img, X_msw_raw_clin, y_msw_raw, X_msw_un_img, _ = get_mixed_data(dataset_msw, 1, factor=1, aug=None)
_, _, _, X_chh_un_img, _ = get_mixed_data(dataset_chh) # Ignored X_chh_un_clin; it shouldn't exist here

# 2. Fit PCA on REAL anatomy only
all_real_imgs = np.vstack([X_msw_raw_img, X_msw_un_img, X_chh_un_img])
img_scaler = StandardScaler().fit(all_real_imgs)
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0).fit(img_scaler.transform(all_real_imgs))

# --- SAFE SCALING IMPLEMENTATION ---
# 3. Define basic feature processing (PCA only)


# 4. Generate the AUGMENTED training set (MSW Labeled)
X_tr_aug_img, X_tr_aug_clin, y_tr_aug, _, _ = get_mixed_data(dataset_msw, NEG_OVERSAMPLE_FACTOR, AUG_FACTOR, qsm_aug)

# 5. Fit the "Patient-Only" Clinical Scaler
# We ONLY fit on real clinical vectors to prevent zeros from skewing the mean/std
clin_scaler = StandardScaler().fit(X_tr_aug_clin)

# 6. Transform Labeled sets into Aligned Space (128 PCA + 6 Clinical)
# We do NOT use feat_scaler on unlabeled data because it has no clinical vector.
X_tr_proc_pca = get_anatomy_only(X_tr_aug_img,pca,img_scaler)
X_tr_proc_clin = clin_scaler.transform(X_tr_aug_clin)
X_tr_final = np.hstack([X_tr_proc_pca, X_tr_proc_clin]) 

# Unlabeled data for Phase 1 (Strictly 128-dim anatomy)
X_msw_un_final = get_anatomy_only(X_msw_un_img,pca,img_scaler)
X_chh_un_final = get_anatomy_only(X_chh_un_img,pca,img_scaler)

y_tr = y_tr_aug 

# 7. Spatial-specific variables (Pixels + Scaled Clinical)
X_tr_spatial_final = np.hstack([X_tr_aug_img, X_tr_proc_clin])
# For unlabeled spatial, we strictly keep anatomy; clinical part is omitted or 
# handled by skip_clin during pretraining to avoid zero-skew.
X_chh_un_spatial = X_chh_un_img 

# 8. Prep Test Set (Aligned)
X_te_img_list, X_te_clin_list, y_te_list = [], [], []
for i in range(len(dataset_chh)):
    img, clin, lbl, _ = dataset_chh[i]
    if lbl != -1:
        img_np = img.numpy().squeeze()
        if img_np.ndim == 3: img_np = img_np[:, :, img_np.shape[2]//2]
        X_te_img_list.append(img_np.flatten()); X_te_clin_list.append(clin.numpy()); y_te_list.append(lbl)

X_te_img_np = np.array(X_te_img_list)
X_te_clin_np = np.array(X_te_clin_list)

X_test_final = np.hstack([get_anatomy_only(X_te_img_np,pca,img_scaler), clin_scaler.transform(X_te_clin_np)])
X_test_spatial_final = np.hstack([X_te_img_np, clin_scaler.transform(X_te_clin_np)])
y_test = np.array(y_te_list)
num_clin = X_te_clin_np.shape[1]

# ============================================================
# PHASED PRETRAINING (ANATOMY -> CLINICAL ALIGNMENT)
# ============================================================

# Models
base_v = SupervisedManifoldVAE(N_PCA_COMPONENTS + num_clin).to(device)
base_va = SupervisedManifoldVAE(N_PCA_COMPONENTS + num_clin).to(device)
base_attn = GatedSpectralAttention(N_PCA_COMPONENTS, n_clinical=num_clin).to(device)
base_vs = SupervisedManifoldVAE(N_PCA_COMPONENTS + num_clin).to(device)
base_at_spatial = EnhancedSpatialAttention(img_dim=64, n_clinical=num_clin, target_dim=N_PCA_COMPONENTS).to(device)

mse = nn.MSELoss()

# --- PREPARE DATA STREAMS ---
# Phase 1 Pool: Everyone's anatomy, strictly 128-dim PCA (Spectral) or 4096-dim Pixels (Spatial)
all_anat_spectral = np.vstack([X_tr_final[:, :N_PCA_COMPONENTS], X_msw_un_final, X_chh_un_final])
all_anat_spatial = np.vstack([X_tr_spatial_final[:, :4096], X_chh_un_spatial]) # Pixels only

# Phase 2 Pool: Aligned MSW Labeled Data ONLY (134-dim)
X_aligned_spectral = torch.tensor(X_tr_final, dtype=torch.float32).to(device)
X_aligned_spatial = torch.tensor(X_tr_spatial_final, dtype=torch.float32).to(device)

# Loaders for Phase 1
pre_loader_anat_spec = DataLoader(TensorDataset(torch.tensor(all_anat_spectral, dtype=torch.float32).to(device)), batch_size=N_PCA_COMPONENTS, shuffle=True)
pre_loader_anat_spat = DataLoader(TensorDataset(torch.tensor(all_anat_spatial, dtype=torch.float32).to(device)), batch_size=N_PCA_COMPONENTS, shuffle=True)

# 1. PHASE 1: ANATOMY RECONSTRUCTION (ALL DATA, NO CLINICAL)
print("PHASE 1: Pretraining on Anatomy Only...")
opt_anat = optim.Adam(list(base_v.parameters()) + list(base_va.parameters()) + 
                      list(base_attn.parameters()) + list(base_vs.parameters()) + 
                      list(base_at_spatial.parameters()), lr=1e-3)

for _ in range(30):
    # Spectral Anatomy
    for b in pre_loader_anat_spec:
        clean = b[0]
        opt_anat.zero_grad()
        
        # Ens Mani VAE path (Requires padding at VAE input, but recon target is clean)
        clean_pad = torch.cat([clean, torch.zeros((clean.size(0), num_clin)).to(device)], dim=1)
        rec_v, mu_v, logv_v = base_v(clean_pad + torch.randn_like(clean_pad)*0.05, mode='pretrain')
        loss_v = mse(rec_v, clean_pad) + 0.01*(-0.5*torch.sum(1+logv_v-mu_v.pow(2)-logv_v.exp())/clean.size(0))
        
        # Spectral ViT path (skip_clin=True bypasses clinical tokens)
        v_img = base_attn(clean + torch.randn_like(clean)*0.05, skip_clin=True)
        v_padded = torch.cat([v_img, torch.zeros((v_img.size(0), num_clin)).to(device)], dim=1)
        rec_va, mu_va, logv_va = base_va(v_padded, mode='pretrain')
        loss_va = mse(rec_va, v_padded.detach()) + 0.01*(-0.5*torch.sum(1+logv_va-mu_va.pow(2)-logv_va.exp())/v_padded.size(0))
        
        (loss_v + loss_va).backward(); opt_anat.step()
        
    # Spatial Anatomy
    for b in pre_loader_anat_spat:
        clean_pix = b[0]
        opt_anat.zero_grad()
        v_spat_img = base_at_spatial(clean_pix + torch.randn_like(clean_pix)*0.05, skip_clin=True)
        v_spat_padded = torch.cat([v_spat_img, torch.zeros((v_spat_img.size(0), num_clin)).to(device)], dim=1)
        rec_vs, mu_vs, logv_vs = base_vs(v_spat_padded, mode='pretrain')
        loss_vs = mse(rec_vs, v_spat_padded.detach()) + 0.01*(-0.5*torch.sum(1+logv_vs-mu_vs.pow(2)-logv_vs.exp())/v_spat_padded.size(0))
        loss_vs.backward(); opt_anat.step()

# 2. PHASE 2: CLINICAL ALIGNMENT (LABELED MSW ONLY)
print("PHASE 2: Pretraining on Clinical Alignment (Labeled Subjects Only)...")
pre_loader_aligned = DataLoader(TensorDataset(X_aligned_spectral, X_aligned_spatial), batch_size=64, shuffle=True)

opt_align = optim.Adam(list(base_va.parameters()) + list(base_attn.parameters()) + 
                       list(base_vs.parameters()) + list(base_at_spatial.parameters()), lr=1e-4)

for _ in range(20):
    for b_spec, b_spat in pre_loader_aligned:
        opt_align.zero_grad()
        
        # Spectral Alignment: Now skip_clin=False, using real clinical vectors
        refined_spec = base_attn(b_spec, skip_clin=False)
        rec_a, mu_a, logv_a = base_va(refined_spec, mode='pretrain')
        loss_spec = mse(rec_a, b_spec) + 0.01*(-0.5*torch.sum(1+logv_a-mu_a.pow(2)-logv_a.exp())/b_spec.size(0))
        
        # Spatial Alignment: skip_clin=False
        refined_spat = base_at_spatial(b_spat, skip_clin=False)
        rec_s, mu_s, logv_s = base_vs(refined_spat, mode='pretrain')
        loss_spat = mse(rec_s, refined_spat.detach()) + 0.01*(-0.5*torch.sum(1+logv_s-mu_s.pow(2)-logv_s.exp())/b_spat.size(0))
        
        (loss_spec + loss_spat).backward(); opt_align.step()
        
# ============================================================
# SYNCED SUBJECT-LEVEL MAPPING (FIXED)
# ============================================================
unique_subject_indices = np.arange(len(dataset_msw))
orig_labels = np.array([dataset_msw[i][2] for i in range(len(dataset_msw))])

subj_map = []
for idx in range(len(dataset_msw)):
    _, _, lbl, _ = dataset_msw[idx]
    
    # We only map labeled data because X_tr_final only contains labeled data
    if lbl != -1:
        reps = NEG_OVERSAMPLE_FACTOR if lbl == 0 else 1
        total_samples = reps * AUG_FACTOR
        for _ in range(total_samples):
            subj_map.append(idx)

subj_map = np.array(subj_map)

# Safety check: Ensure the mapping matches the augmented feature matrix
if len(subj_map) != X_tr_final.shape[0]:
    print(f"CRITICAL: Mapping ({len(subj_map)}) != Features ({X_tr_final.shape[0]})")
    subj_map = subj_map[:X_tr_final.shape[0]]

print(f"DEBUG: Feature matrix shape: {X_tr_final.shape[0]}")
print(f"DEBUG: Subject map shape: {subj_map.shape[0]}")

# Emergency alignment (if floating point or rounding issues occur)
if len(subj_map) > len(X_tr_final):
    subj_map = subj_map[:len(X_tr_final)]

# ============================================================
# 3. LEAK-FREE K-FOLD LOOP
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
fold_preds_v, fold_preds_va, fold_preds_vs = [], [], []
fold_ths_v, fold_ths_va, fold_ths_vs = [], [], []

NEG_WEIGHT = 12

for fold, (train_subj_idx, val_subj_idx) in enumerate(skf.split(unique_subject_indices, orig_labels)):
    train_mask = np.isin(subj_map, train_subj_idx)
    val_mask = np.isin(subj_map, val_subj_idx)
    
    X_f_train_pca = X_tr_final[train_mask]
    X_f_val_pca = X_tr_final[val_mask]
    X_f_train_spatial = X_tr_spatial_final[train_mask]
    X_f_val_spatial = X_tr_spatial_final[val_mask]
    y_f_train = y_tr[train_mask]
    y_f_val = y_tr[val_mask]

    m_v, m_va, m_at = copy.deepcopy(base_v), copy.deepcopy(base_va), copy.deepcopy(base_attn)
    m_vs, m_ats = copy.deepcopy(base_vs), copy.deepcopy(base_at_spatial)

    ft_loader = DataLoader(TensorDataset(
        torch.tensor(X_f_train_pca, dtype=torch.float32).to(device),
        torch.tensor(X_f_train_spatial, dtype=torch.float32).to(device),
        torch.tensor(y_f_train, dtype=torch.float32).to(device)
    ), batch_size=64, shuffle=True)

    opt_v = optim.Adam(m_v.parameters(), lr=1e-5, weight_decay=1e-2)
    opt_va = optim.Adam([{'params': m_va.parameters(), 'lr': 2e-5}, {'params': m_at.parameters(), 'lr': 8e-5}], weight_decay=5e-2)
    opt_vs = optim.Adam([{'params': m_vs.parameters(), 'lr': 2e-5}, {'params': m_ats.parameters(), 'lr': 8e-5}], weight_decay=5e-2)

    for _ in range(50):
        for b_pca, b_spatial, by in ft_loader:
            # 1. Create a mask of the same shape as the clinical part [Batch, 6]
            # p=0.2 means each clinical feature has a 20% chance of being zeroed
            clin_dropout_p = 0.3 
            
            # Mask for PCA-based models (last 6 indices)
            clin_mask = (torch.rand(b_pca.size(0), 6) > clin_dropout_p).float().to(device)
            
            b_pca_aug = b_pca.clone()
            # Multiply clinical section by the mask
            b_pca_aug[:, N_PCA_COMPONENTS:] = b_pca_aug[:, N_PCA_COMPONENTS:] * clin_mask
            
            # Mask for Spatial model (last 6 indices after 4096 pixels)
            b_spatial_aug = b_spatial.clone()
            b_spatial_aug[:, 4096:] = b_spatial_aug[:, 4096:] * clin_mask
            
            # --- Continue training as normal ---
            smoothed_labels = by * 0.95 + (1 - by) * 0.05
            weights = (by == 0).float() * NEG_WEIGHT + 1.0
            
            opt_v.zero_grad()
            # Use the augmented (dropped) batch
            loss_v = nn.functional.binary_cross_entropy(m_v(b_pca_aug, mode='fine_tune'), smoothed_labels, weight=weights)
            loss_v.backward(); opt_v.step()

            opt_va.zero_grad()
            # Use the augmented (dropped) batch
            loss_va = nn.functional.binary_cross_entropy(m_va(m_at(b_pca_aug), mode='fine_tune'), smoothed_labels, weight=weights)
            loss_va.backward(); opt_va.step()

            opt_vs.zero_grad()
            # Use the augmented (dropped) batch
            loss_vs = nn.functional.binary_cross_entropy(m_vs(m_ats(b_spatial_aug), mode='fine_tune'), smoothed_labels, weight=weights)
            loss_vs.backward(); opt_vs.step()

    fold_ths_v.append(calibrate_balanced(m_v, None, X_f_val_pca, y_f_val, device))
    fold_ths_va.append(calibrate_balanced(m_va, m_at, X_f_val_pca, y_f_val, device))
    fold_ths_vs.append(calibrate_balanced(m_vs, m_ats, X_f_val_spatial, y_f_val, device))
    
    with torch.no_grad():
        t_te_pca = torch.tensor(X_test_final, dtype=torch.float32).to(device)
        t_te_spatial = torch.tensor(X_test_spatial_final, dtype=torch.float32).to(device)
        
        fold_preds_v.append(m_v(t_te_pca, mode='fine_tune').cpu().numpy())
        fold_preds_va.append(m_va(m_at(t_te_pca), mode='fine_tune').cpu().numpy())
        fold_preds_vs.append(m_vs(m_ats(t_te_spatial), mode='fine_tune').cpu().numpy())
        
pv, pva, pvs = np.mean(fold_preds_v, axis=0), np.mean(fold_preds_va, axis=0), np.mean(fold_preds_vs, axis=0)
th_v, th_va, th_vs = np.mean(fold_ths_v), np.mean(fold_ths_va), np.mean(fold_ths_vs)

lr_clin = LogisticRegression(class_weight='balanced', random_state=0)
lr_clin.fit(X_tr_proc_clin, y_tr)
p_lr_clin = lr_clin.predict_proba(clin_scaler.transform(X_te_clin_np))[:, 1]

# --- 2. Clinical + PCA (The Linear "Spectral" Baseline) ---
lr_combined = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=0)
lr_combined.fit(X_tr_final, y_tr) # X_tr_final is already [PCA + Clin]
p_lr_comb = lr_combined.predict_proba(X_test_final)[:, 1]

# --- 3. Evaluate at 0.5 Threshold ---
res_lr_clin = get_m(y_test, (p_lr_clin >= 0.5), p_lr_clin)
res_lr_comb = get_m(y_test, (p_lr_comb >= 0.5), p_lr_comb)

# 1. Ensure the order matches exactly: LR Clin, LR PCA+Clin, VAE, Spatial, Spectral
res = [
    res_lr_clin,                        # Clinical LR (at 0.5)
    res_lr_comb,                        # LR PCA+Clin (at 0.5)
    get_m(y_test, (pv >= th_v), pv),    # VAE (at th_v)
    get_m(y_test, (pvs >= th_vs), pvs), # Spatial ViT (at th_vs)
    get_m(y_test, (pva >= th_va), pva)  # Spectral ViT (at th_va)
]

# 2. Match the thresholds to that same order
final_thresholds = [0.5, 0.5, th_v, th_vs, th_va]

names = ['LR (Clin Only)', 'LR (PCA+Clin)', 'Ens Mani VAE', 'Spatial ViT', 'Spectral ViT']

# 3. Print headers and metrics
print(f"\n{'Metric':<15} | " + " | ".join([f"{n:<15}" for n in names]))
print("-" * 115)
for i, m_name in enumerate(['Accuracy', 'Sensitivity', 'Specificity', 'AUC']):
    row_values = " | ".join([f"{r[i]:<15.4f}" for r in res])
    print(f"{m_name:<15} | {row_values}")
print("-" * 115)

# 4. Your corrected Threshold line
row_thresholds = " | ".join([f"{t:<15.4f}" for t in final_thresholds])
print(f"{'Avg Threshold':<15} | {row_thresholds}")


Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


Using device: cuda:2

Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.

In [2]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

def calibrate_high_specificity(model, attn, X_v, y_v, target_spec=0.70, batch_size=32):
    model.eval()
    if attn: attn.eval()
    
    all_p = []
    with torch.no_grad():
        for i in range(0, len(X_v), batch_size):
            x_batch = torch.tensor(X_v[i:i+batch_size], dtype=torch.float32).to(device)
            feat = attn(x_batch) if attn else x_batch
            all_p.append(model(feat, mode='fine_tune').cpu())
    
    probs = torch.cat(all_p).numpy()
    
    best_t = 0.5
    best_metric = -1
    
    # Search for the threshold that maximizes Sensitivity + (2 * Specificity)
    # This weighting forces the model to care more about the minority class
    for t in np.linspace(0.01, 0.95, 200):
        preds = (probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_v, preds).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # We use a weighted Youden Index to favor Specificity
        weighted_metric = ((1/12) * sens) + (12.0 * spec) 
        
        if weighted_metric > best_metric:
            best_metric = weighted_metric
            best_t = t
            
    return best_t

# --- RUN HIGH-SPECIFICITY INFERENCE ---
print("\n--- Running High-Specificity Calibration ---")
th_v_hs = calibrate_high_specificity(m_v, None, X_f_val_pca, y_f_val)
th_va_hs = calibrate_high_specificity(m_va, m_at, X_f_val_pca, y_f_val)
th_vs_hs = calibrate_high_specificity(m_vs, m_ats, X_f_val_spatial, y_f_val)

# Final Evaluation Function
def get_metrics_hs(y_true, y_prob, threshold):
    preds = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    acc = (tp + tn) / len(y_true)
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    auc = roc_auc_score(y_true, y_prob)
    return [acc, sens, spec, auc]

res_hs = [
    get_metrics_hs(y_test, pv, th_v_hs),
    get_metrics_hs(y_test, pva, th_va_hs),
    get_metrics_hs(y_test, pvs, th_vs_hs)
]

print(f"\n{'Metric':<15} | {'Ens Mani VAE':<15} | {'Spectral ViT':<15} | {'Spatial ViT':<15}")
print("-" * 70)
for i, m_name in enumerate(['Accuracy', 'Sensitivity', 'Specificity', 'AUC']):
    row = " | ".join([f"{r[i]:<15.4f}" for r in res_hs])
    print(f"{m_name:<15} | {row}")
print("-" * 70)
print(f"{'New Threshold':<15} | {th_v_hs:<15.4f} | {th_va_hs:<15.4f} | {th_vs_hs:<15.4f}")


--- Running High-Specificity Calibration ---

Metric          | Ens Mani VAE    | Spectral ViT    | Spatial ViT    
----------------------------------------------------------------------
Accuracy        | 0.2924          | 0.7459          | 0.8825         
Sensitivity     | 0.2471          | 0.7541          | 0.9567         
Specificity     | 0.8056          | 0.6528          | 0.0417         
AUC             | 0.5242          | 0.7585          | 0.7235         
----------------------------------------------------------------------
New Threshold   | 0.3879          | 0.2604          | 0.1186         
